In [0]:
# Cấp quyền cho Spark có thể chui vào Data Lake của em
spark.conf.set(
    f"fs.azure.account.key.{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net",
    STORAGE_ACCOUNT_KEY
)

# Đường dẫn dùng giao thức chuẩn abfss:// (Azure Blob File System Secure)
BRONZE_TABLE_PATH = f"abfss://{CONTAINER_NAME}@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/bronze/events"
CHECKPOINT_PATH = f"abfss://{CONTAINER_NAME}@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/checkpoints/bronze_events"

In [0]:
from pyspark.sql.functions import current_timestamp, expr

print("Đang đọc luồng dữ liệu từ Event Hubs...")

raw_stream_df = (spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP_SERVERS)
    .option("subscribe", EH_TOPIC)
    .option("kafka.sasl.mechanism", "PLAIN")
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.jaas.config", EH_SASL)
    .option("startingOffsets", "earliest") # Đọc lại toàn bộ data cũ để test
    .load()
)

# Bung dữ liệu nhị phân thành chuỗi string và thêm cột thời gian
bronze_df = raw_stream_df.select(
    expr("cast(value as string) as raw_payload"),
    current_timestamp().alias("ingestion_timestamp")
)

In [0]:
print(f"Đang ghi dữ liệu xuống Data Lake tại: {BRONZE_TABLE_PATH}")

query = (bronze_df.writeStream
    .format("delta") # Lưu theo định dạng Delta Lake chuẩn mực
    .outputMode("append") # Chỉ thêm data mới, không sửa/xóa data cũ (Append-only)
    .option("checkpointLocation", CHECKPOINT_PATH) # Sổ ghi chép chống sập nguồn
    .start(BRONZE_TABLE_PATH)
)

# Hàm này giúp Notebook giữ cho cái luồng chạy liên tục không bị ngắt
# (Em có thể bỏ qua nếu muốn mở Cell khác chạy tiếp)
# query.awaitTermination()